In [ ]:
# reconstruct_timings.ipynb -- recover per-combo training time from on-disk
# byproducts (nothing recorded it explicitly; this mines what exists):
#   * status.json ts  -> last-claim start;  done.json done_ts -> finish
#     => attempt_wall_s (last attempt only; resumes reset the claim)
#   * probe_ckpts/epNNN.pt mtimes -> a timestamp every probe epoch, SURVIVES
#     resumes => per-epoch pace. Interruption gaps appear as outlier deltas,
#     so the MEDIAN pace x total epochs estimates ACTIVE training time.
# Writes <out_dir>/combo_timings.json (one row per done combo, with axes) and
# prints family summaries. Pull locally via tools/sync_results.ps1.
import json, statistics, sys, time
from pathlib import Path

REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
EPOCHS = 30

if REPO not in sys.path:
    sys.path.insert(0, REPO)
import copy
from VICReg_review.sweep.config import SweepConfig

root = Path(REPO) / OUT_DIR
cfg = SweepConfig.load(f'{REPO}/VICReg_review/sweep/sweep.yaml')
full = copy.deepcopy(cfg)
full.grid.exclude = []
by_id = {c.combo_id: c for c in full.iter_combos()}


def _read(p):
    try:
        return json.loads(Path(p).read_text(encoding='utf-8'))
    except Exception:
        return None


rows = []
for d in sorted(root.iterdir()):
    c = by_id.get(d.name)
    if c is None or not d.is_dir():
        continue
    man = _read(d / 'vicreg_review_h5_manifest.json') or {}
    done = _read(d / 'done.json') or {}
    is_done = bool(done) or man.get('status') == 'done'
    if not is_done:
        continue
    st = _read(d / 'status.json') or {}
    probes = sorted(
        (int(f.stem[2:]), f.stat().st_mtime)
        for f in (d / 'probe_ckpts').glob('ep*.pt')
        if f.stem[2:].isdigit()
    ) if (d / 'probe_ckpts').is_dir() else []

    done_ts = done.get('done_ts')
    claim_ts = st.get('ts')
    attempt_wall = (done_ts - claim_ts) if (done_ts and claim_ts and done_ts > claim_ts) else None

    # median per-epoch pace from consecutive probe deltas (gap-robust)
    paces = []
    for (e0, t0), (e1, t1) in zip(probes, probes[1:]):
        if e1 > e0 and t1 > t0:
            paces.append((t1 - t0) / (e1 - e0))
    med_epoch_s = statistics.median(paces) if paces else None
    est_active_s = med_epoch_s * EPOCHS if med_epoch_s else None
    span_s = (probes[-1][1] - probes[0][1]) if len(probes) >= 2 else None

    rows.append({
        'combo_id': d.name,
        'arm': c.arm, 'output_dim': c.output_dim, 'num_latents': c.num_latents,
        'view': c.view, 'train_games': c.train_games,
        'done_by': done.get('vm') or man.get('status'),
        'done_ts': done_ts,
        'last_claim_ts': claim_ts,
        'attempt_wall_s': round(attempt_wall) if attempt_wall else None,
        'probe_points': len(probes),
        'median_epoch_s': round(med_epoch_s, 1) if med_epoch_s else None,
        'est_active_wall_s': round(est_active_s) if est_active_s else None,
        'probe_span_s': round(span_s) if span_s else None,
    })

out = root / 'combo_timings.json'
tmp = out.with_name(f'{out.name}.tmp.{time.time_ns()}')
tmp.write_text(json.dumps({'created_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
                           'epochs': EPOCHS, 'rows': rows},
                          ensure_ascii=False, indent=1), encoding='utf-8')
tmp.replace(out)
print(f'{len(rows)} done combos timed -> {out}')

with_est = [r for r in rows if r['est_active_wall_s']]
print(f'  with pace estimate: {len(with_est)} '
      f'(need >=2 probe snapshots; the rest have attempt_wall only)')

def fam_key(r):
    lat = r['num_latents']
    return f"n{r['train_games']}_view{int(r['view'] * 100)}_lat{lat}"

fams = {}
for r in with_est:
    fams.setdefault(fam_key(r), []).append(r['est_active_wall_s'])
print()
print(f"{'cell (n_view_lat)':28} {'combos':>6} {'median active':>14} {'min..max':>19}")
for k in sorted(fams, key=lambda k: -statistics.median(fams[k])):
    v = fams[k]
    med = statistics.median(v)
    print(f'{k:28} {len(v):6d} {med / 3600:11.2f} h  '
          f'{min(v) / 3600:7.2f}..{max(v) / 3600:.2f} h')
